In [1]:
from selenium import webdriver
import undetected_chromedriver as uc # 模拟用户行为，躲避人机验证
# import BeautifulSoup
from bs4 import BeautifulSoup
import time
import os

#### 访问网页并下载

In [2]:
url = "https://www.zillow.com/stanford-ca/sold"
html_path = './zillow.html'

# chrome_options = webdriver.ChromeOptions() #Headless means open a browser without Graphic User Interface GUI
# chrome_options.headless = True
# chrome = webdriver.Chrome(
#     options = chrome_options
# ) # Opens the URL page using headless chrome, returns None
# page = chrome.get(url)
# html_content = chrome.page_source

if not os.path.exists(html_path):
    driver = uc.Chrome()
    driver.get(url)
    html_content = driver.page_source

    time.sleep(2)
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

#### 抓取房子id

In [3]:
page = BeautifulSoup(open(html_path, 'r'))
# print(page.prettify()) # Extract data
# links = [a['herf'] for a in page.find_all(
#     'a', ' list-card'
# )]
# ids = [l.split('/')[-2].split('_')[0]
#        for l in links]
# print(links)
# print(ids)
cards = page.find_all('article', id=lambda x: x and x.startswith('zpid_'))

print(f"共发现 {len(cards)} 个房源卡片")

links = []
ids = []
for card in cards:
    # 提取 ID (去掉 'zpid_' 前缀)
    house_id = card.get('id').replace('zpid_', '')
    ids += [house_id]
    # 提取链接
    # 链接通常在 data-test="property-card-link" 的 a 标签里
    link_tag = card.find('a', {'data-test': 'property-card-link'})
    if link_tag:
        link = link_tag.get('href')
        # 如果是相对路径，拼上域名
        if link.startswith('/'):
            link = 'https://www.zillow.com' + link
        links += [link]
    else:
        link = "未找到链接"
    
    print(f"ID: {house_id} | 链接: {link}")

共发现 28 个房源卡片
ID: 19507429 | 链接: https://www.zillow.com/homedetails/72-Peter-Coutts-Cir-Stanford-CA-94305/19507429_zpid/
ID: 19506859 | 链接: https://www.zillow.com/homedetails/843-Sonoma-Ter-Stanford-CA-94305/19506859_zpid/
ID: 19507138 | 链接: https://www.zillow.com/homedetails/866-Tolman-Dr-Palo-Alto-CA-94305/19507138_zpid/
ID: 19507014 | 链接: https://www.zillow.com/homedetails/827-Lathrop-Dr-Stanford-CA-94305/19507014_zpid/
ID: 19507240 | 链接: https://www.zillow.com/homedetails/801-Allardice-Way-Stanford-CA-94305/19507240_zpid/
ID: 19507422 | 链接: https://www.zillow.com/homedetails/65-Peter-Coutts-Cir-Stanford-CA-94305/19507422_zpid/
ID: 19507030 | 链接: https://www.zillow.com/homedetails/835-Mayfield-Ave-Stanford-CA-94305/19507030_zpid/
ID: 19507385 | 链接: https://www.zillow.com/homedetails/28-Peter-Coutts-Cir-Stanford-CA-94305/19507385_zpid/
ID: 19507340 | 链接: https://www.zillow.com/homedetails/64-Pearce-Mitchell-Pl-Stanford-CA-94305/19507340_zpid/
ID: 19507369 | 链接: https://www.zillow.com/

#### Extract Data

In [4]:
# # Extract Data
# sold_items = [a.txt for a in page.find(
#     'div', 'ds-home-detals-chip'
# ).find('p').find_all('span')]
# for item in sold_items:
#     if 'Sold:' in item:
#         result['Sold Price'] = item.split(' ')[1]
#     if 'Sold on' in item:
#         result['Sold on'] = item.split(' ')[-1]

# 初始化一个列表来存放所有房子的结果
all_results = []

# 1. 找到所有房屋卡片
cards = page.find_all('article', id=lambda x: x and x.startswith('zpid_'))

for card in cards:
    result = {} # 为每个房子创建一个新字典
    
    # 提取 ID
    result['ID'] = card.get('id').replace('zpid_', '')
    
    # 提取链接
    link_tag = card.find('a', {'data-test': 'property-card-link'})
    result['Link'] = link_tag.get('href') if link_tag else "N/A"
    
    # --- 关键修改：从卡片内部提取价格和状态 ---
    
    # 提取价格 (data-test="property-card-price")
    price_tag = card.find('span', {'data-test': 'property-card-price'})
    result['Price'] = price_tag.get_text(strip=True) if price_tag else "N/A"
    
    # 提取状态 (例如 "Sold", "For Sale")
    # Zillow 的状态通常在详情文字的末尾或特定的 span 里
    card_text = card.get_text(separator='|', strip=True)
    if 'Sold' in card_text:
        result['Status'] = 'Sold'
    elif 'For sale' in card_text:
        result['Status'] = 'For Sale'
    else:
        result['Status'] = 'Unknown'

    # 提取图片链接
    img_tag = card.find('img')
    img_url = "N/A"
    if img_tag:
        # 尝试获取不同的属性，因为 Zillow 经常变换懒加载属性名
        img_url = img_tag.get('src') or img_tag.get('data-src')
        
        # 如果 src 是 base64 加密的占位图（以 data:image 开头），则尝试获取 srcset
        if img_url and img_url.startswith('data:image'):
            srcset = img_tag.get('srcset')
            if srcset:
                # srcset 通常包含多个分辨率，取第一个即可
                img_url = srcset.split(' ')[0]
    result['Image URL'] = img_url


    all_results.append(result)

# 打印所有结果
import pprint
pprint.pprint(all_results)

[{'ID': '19507429',
  'Image URL': 'https://photos.zillowstatic.com/fp/6e76332caba0646a6f258a7ba86b53ab-p_e.jpg',
  'Link': 'https://www.zillow.com/homedetails/72-Peter-Coutts-Cir-Stanford-CA-94305/19507429_zpid/',
  'Price': '$1.15M',
  'Status': 'Sold'},
 {'ID': '19506859',
  'Image URL': 'https://photos.zillowstatic.com/fp/760fe83c5950dd1830db5b901d42d85f-p_e.jpg',
  'Link': 'https://www.zillow.com/homedetails/843-Sonoma-Ter-Stanford-CA-94305/19506859_zpid/',
  'Price': '$3.00M',
  'Status': 'Sold'},
 {'ID': '19507138',
  'Image URL': 'https://photos.zillowstatic.com/fp/eb05af5a57478a2b55d6b3fc9f7d612f-p_e.jpg',
  'Link': 'https://www.zillow.com/homedetails/866-Tolman-Dr-Palo-Alto-CA-94305/19507138_zpid/',
  'Price': '$2.80M',
  'Status': 'Sold'},
 {'ID': '19507014',
  'Image URL': 'https://photos.zillowstatic.com/fp/31163e3410e08b3ad4c4b815003890e1-p_e.jpg',
  'Link': 'https://www.zillow.com/homedetails/827-Lathrop-Dr-Stanford-CA-94305/19507014_zpid/',
  'Price': '$3.80M',
  'Statu

### 保存结果

In [5]:
import pandas as pd

if all_results:
    df = pd.DataFrame(all_results)
    df.to_csv('zillow_data.csv', index=False, encoding='utf-8-sig')
    print(f"成功抓取 {len(all_results)} 条数据并保存至 zillow_data.csv")
else:
    print("未抓取到数据，请检查 HTML 源码是否包含房源卡片。")

成功抓取 28 条数据并保存至 zillow_data.csv


# 抓取图片

In [6]:
import os
import time
import requests
import pandas as pd
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import random

def download_house_images(house_id, house_url, driver):
    """进入详情页并下载所有图片"""
    print(f"正在处理房源 ID: {house_id}...")
    
    save_dir = f'zillow_data/images/{house_id}'
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    else:
        print(f"文件夹 {house_id} 已存在，跳过。")
        return []

    # --- 1. 访问页面 ---
    max_retries = 3
    for i in range(max_retries):
        try:
            driver.get(house_url)
            time.sleep(random.uniform(3, 5)) # 等待基础加载
            break 
        except Exception as e:
            if i == max_retries - 1: return []
            time.sleep(5)

    # --- 2. 模拟滚动触发懒加载 ---
    # Zillow 的图片通常随着滚动加载，甚至需要多次滚动
    for _ in range(5): 
        driver.execute_script("window.scrollBy(0, 1500);")
        time.sleep(1.5)

    # --- 3. 解析页面 ---
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # 修改定位逻辑：找到所有包含 picture 的按钮或列表项
    # Zillow 的图片通常在特定的 data-mst-cy="hollywood-photo-carousel" 属性里
    picture_tags = soup.find_all('picture')
    
    img_urls = []
    # 设置 Request Headers 防止下载图片时被封
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
        "Referer": "https://www.zillow.com/"
    }

    for idx, pic in enumerate(picture_tags):
        url = None
        
        # 尝试从 <source type="image/jpeg"> 的 srcset 中提取最低清地址
        source_jpeg = pic.find('source', {'type': 'image/jpeg'})
        if source_jpeg and source_jpeg.get('srcset'):
            # srcset 格式: "url1 192w, url2 384w, url3 1536w"
            # 我们取逗号分隔的第一项，再取空格前的 URL
            srcset_list = source_jpeg['srcset'].split(',')
            best_choice = srcset_list[0].strip().split(' ')[0]
            url = best_choice
        
        # 如果没找到 source，退而求其次找 img 标签的 src
        if not url:
            img_tag = pic.find('img')
            if img_tag:
                url = img_tag.get('src')

        if url and url.startswith('http'):
            # 过滤掉一些小的 icon 或占位图（Zillow 房源图通常包含 'fp/'）
            if 'zillowstatic.com/fp/' not in url:
                continue

            img_urls.append(url)
            
            # --- 4. 执行下载 ---
            try:
                # 使用带 headers 的 requests
                response = requests.get(url, headers=headers, timeout=15)
                if response.status_code == 200:
                    with open(f"{save_dir}/{idx+1}.jpg", 'wb') as f:
                        f.write(response.content)
                else:
                    print(f"图片 {idx+1} 下载返回状态码: {response.status_code}")
                
                # 稍微停顿，避免下载过快
                time.sleep(random.uniform(0.5, 1.2))
            except Exception as e:
                print(f"图片 {idx+1} 下载出错: {e}")

    print(f"房源 {house_id} 实际解析并下载了 {len(img_urls)} 张图片")
    return img_urls


In [7]:
driver = uc.Chrome()
for house in all_results:
    h_id = house['ID']
    h_url = house['Link']
    
    if h_url != "N/A":
        downloaded_urls = download_house_images(h_id, h_url, driver)
        house['Local_Image_Folder'] = f'zillow_data/images/{h_id}'
        house['Image_Count'] = len(downloaded_urls)
        
        # 每处理完一个房子，休息一下，防止被封
        time.sleep(random.uniform(4, 8)) # 随机数让行为更像真实用户

# 3. 更新并保存 CSV 索引文件
df = pd.DataFrame(all_results)
df.to_csv('zillow_data/properties_metadata.csv', index=False, encoding='utf-8-sig')

driver.quit()

正在处理房源 ID: 19507429...
文件夹 19507429 已存在，跳过。
正在处理房源 ID: 19506859...
文件夹 19506859 已存在，跳过。
正在处理房源 ID: 19507138...
文件夹 19507138 已存在，跳过。
正在处理房源 ID: 19507014...
文件夹 19507014 已存在，跳过。


KeyboardInterrupt: 